In [1]:
# Some reference is taken from here: https://captum.ai/tutorials/Bert_SQUAD_Interpret

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split

from transformers import BertModel, BertTokenizer

from sklearn.preprocessing import LabelEncoder

from captum.attr import LayerIntegratedGradients
from captum.attr import visualization as viz

import time
from tqdm import tqdm

import pickle

from utils import *

/home/fahim/anaconda3/envs/explainable-news/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2023-03-16 09:10:03.710383: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-03-16 09:10:04.122944: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda-11.7/lib64:/usr/local/cuda-11.7/lib64
2023-03-16 09:10:04.122986: W tensorflow/compiler/xla/stream_executor/

In [2]:
label_enc = LabelEncoder()

with open(f'label_encoder.pkl', 'rb')as f:
    
    label_enc = pickle.load(f)
    
    f.close()

/home/fahim/anaconda3/envs/explainable-news/lib/python3.8/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelEncoder from version 1.2.1 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
label_enc.classes_

array(['bangladesh', 'economy', 'education', 'entertainment',
       'international', 'life-style', 'opinion', 'sports', 'technology'],
      dtype=object)

In [4]:
bert_model_name = "csebuetnlp/banglabert_large"
bert = BertModel.from_pretrained(bert_model_name)
tokenizer = BertTokenizer.from_pretrained(bert_model_name)

device = 'cpu'

model = ClassifyNews(bert)

saved_model_path = f'saved_weights/1678583057.8259-news-classification-even-data-31.pth'

checkpoint = torch.load(saved_model_path)

model.load_state_dict(checkpoint)

model.to(device)

model.eval()

You are using a model of type electra to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
Some weights of the model checkpoint at csebuetnlp/banglabert_large were not used when initializing BertModel: ['electra.encoder.layer.8.attention.self.key.bias', 'electra.encoder.layer.3.output.dense.bias', 'electra.encoder.layer.19.attention.self.key.weight', 'electra.encoder.layer.18.output.dense.weight', 'electra.encoder.layer.19.intermediate.dense.bias', 'discriminator_predictions.dense_prediction.weight', 'electra.encoder.layer.5.attention.self.key.bias', 'electra.encoder.layer.12.attention.self.key.weight', 'electra.encoder.layer.4.output.dense.weight', 'electra.encoder.layer.17.attention.self.value.weight', 'electra.encoder.layer.21.output.LayerNorm.weight', 'electra.encoder.layer.12.output.dense.weight', 'electra.encoder.layer.11.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder

ClassifyNews(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise

In [28]:
data = pd.read_csv(f'data/even_cleaned_data.csv')

_, test_data = train_test_split(data, test_size = 0.2, random_state = 2023, stratify = data['label'])

In [29]:
test_data_file = NewsDatasets(test_data)

test_loader = torch.utils.data.DataLoader(test_data_file, batch_size=1, shuffle=True)

In [30]:
real_data = next(iter(test_loader))

In [31]:
text, labels = real_data

In [32]:
len(text)

1

In [33]:
labels.shape

torch.Size([1])

In [34]:
labels

tensor([3])

In [35]:
text[0]

'শিল্পকলা একাডেমীর আয়োজন[SEP]শিল্পকলা একাডেমীর জাতীয় নাট্যশালার প্রধান মিলনায়তনে হয়েছে দিনের নজরুল উৎসব নির্ধারিত সময়ের এক ঘণ্টা অনুষ্ঠান একাডেমীর সচিব জাহাঙ্গীর হোসেন চৌধুরীর সভাপতিত্বে আলোচনায় অংশ নেন ভাষাসংগ্রামী আহমদ রফিক অধ্যাপক করুণাময় গোস্বামী আলোচনা শেষে সমবেত নজরুলসংগীত পরিবেশন শিশুশিল্পীরা দ্রোহ সৃষ্টি শীর্ষক সমবেত নৃত্যালেখ্য পরিবেশন একাডেমীর রেপার্টরি নৃত্য দল গান নাচ চলতে পালাক্রমে'

In [36]:
label_enc.inverse_transform(labels.numpy())

array(['entertainment'], dtype=object)

## Create input and baseline for explainability

In [37]:
def input_and_baseline(text):
    
    tokenizer_config = {
        "max_length": 250,
        "truncation": True,
        "add_special_tokens": False
    }
    
    baseline_token_id = tokenizer.pad_token_id 
    sep_token_id = tokenizer.sep_token_id 
    cls_token_id = tokenizer.cls_token_id 

    text_ids = tokenizer.encode(text, **tokenizer_config)
    
    input_ids = [cls_token_id] + text_ids + [sep_token_id]
   
    raw_tokens = tokenizer.convert_ids_to_tokens(input_ids)
  

    baseline_input_ids = [cls_token_id] + [baseline_token_id] * len(text_ids) + [sep_token_id]
    
    return torch.tensor([input_ids], device = 'cpu'), torch.tensor([baseline_input_ids], device = 'cpu'), raw_tokens

In [38]:
input_ids, baseline_input_ids, all_tokens = input_and_baseline(text[0])

In [39]:
input_ids

tensor([[    2, 14230, 25371,     1,     3, 14230, 25371,     1,  7345, 17960,
          1415,     1,     1,  2180,  5428,  3782,  5604,     1,   788,  3938,
          3570, 25371,  4513,  7586,  2131,  7554,  8870,     1,  1550,  3330,
          3079,  9636,  3829,   428,  4782,  6299,  3840,     1, 22747,  2613,
          2284, 13807,  5428, 15292,  8265,  2302,  7295,   825,  2115,  2461,
          2498, 10473, 13807,  8699,  4455,  1962,  8265, 25371, 25853,  2683,
          1080,  8699,  1382,  1755,  5085,  3858,  7427,  7851,     3]])

In [40]:
baseline_input_ids

tensor([[2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3]])

In [41]:
print(" ".join(all_tokens))

[CLS] শিল্পকলা একাডেমীর [UNK] [SEP] শিল্পকলা একাডেমীর [UNK] নাট্য ##শালার প্রধান [UNK] [UNK] দিনের নজরুল উৎসব নির্ধারিত [UNK] এক ঘণ্টা অনুষ্ঠান একাডেমীর সচিব জাহাঙ্গীর হোসেন চৌধুরীর সভাপতিত্বে [UNK] অংশ নেন ভাষা ##সংগ ##্রাম ##ী আহমদ রফিক অধ্যাপক [UNK] গোস্বামী আলোচনা শেষে সমবেত নজরুল ##সংগীত পরিবেশন শিশু ##শিল্পী ##রা দ্র ##োহ সৃষ্টি শীর্ষক সমবেত নৃত্য ##ালে ##খ্য পরিবেশন একাডেমীর রেপ ##ার্ট ##রি নৃত্য দল গান নাচ চলতে পালা ##ক্রমে [SEP]


In [42]:
def model_output(inputs):
    return model(inputs)[0]

In [43]:
lig = LayerIntegratedGradients(model_output, model.bert.embeddings)

In [44]:
attributions, delta = lig.attribute(inputs= input_ids.to(device),
                                    baselines= baseline_input_ids.to(device),
                                    n_steps = 150,
                                    return_convergence_delta=True
                                    )

In [45]:
print(attributions.size())

torch.Size([1, 69, 1024])


In [46]:
torch.argmax(model(input_ids)[0]).numpy()

array(3)

In [47]:
torch.max(model(input_ids)[0])

tensor(4.3191, grad_fn=<MaxBackward1>)

In [48]:
model(input_ids)

tensor([[-9.8977e-01, -5.2120e-01, -1.7661e+00,  4.3191e+00, -3.2605e-01,
          3.6320e-04, -7.8516e-01, -1.4648e-02, -2.6663e-01]],
       grad_fn=<AddmmBackward0>)

In [49]:
def summarize_attributions(attributions):

    attributions = attributions.sum(dim=2).squeeze(0)
    attributions = attributions / torch.norm(attributions)
    attributions = attributions.cpu().detach().numpy()
    
    return attributions

attributions_sum = summarize_attributions(attributions)

In [50]:
score_vis = viz.VisualizationDataRecord(
                        word_attributions = attributions_sum,
                        pred_prob = torch.max(model(input_ids)[0]),
                        pred_class = torch.argmax(model(input_ids)[0]).numpy(),
                        true_class = labels.item(),
                        attr_class = text,
                        attr_score = attributions_sum.sum(),       
                        raw_input_ids = all_tokens,
                        convergence_score = delta)

viz.visualize_text([score_vis])

In [ ]:
from captum.attr import Saliency

In [ ]:
sal = Saliency(model_output_sal)



In [ ]:
def model_output_sal(inputs):
    
    output = model(inputs)
    
    label = torch.argmax(output)
    
    return label

In [ ]:
model_output_sal(input_ids.to(device))

In [ ]:
labels[0]

In [ ]:
atttr, delta = sal.attribute(input_ids.to(device),
                     labels[0])